# SDPA 教学版实现走读


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. Kernel 总体结构（算法伪代码）

```text
输入/输出：[B,S,H]，H=64/128；任务=(batch_id,q_block_id)
Cube 与 Vector 使用相同的逻辑组号及任务映射；两个 AIV 各处理 Q 块的一半行。
对本组负责的每个任务：
    Vector 初始化 m=-inf、l=0、acc=0，并区分“尚无有效列”的行
    对每个需要处理的 KV 块：
        Cube：load Q/K → L0A/L0B → QKᵀ → 把 FP32 s 搬给 Vector
        QK_READY / QK_RELEASE：确保生产完成与下一次覆写安全
        Vector：s *= 1/sqrt(H)；应用按全局位置定义的 causal mask
                合并旧 m 与块最大值，更新 alpha/l；生成未归一化 p
                p 按 PV 所用输入格式转换/搬到 L1
        P_READY / P_RELEASE：确保 Cube 读完前不覆写 p
        Cube：p@V → 将 FP32 pv 搬给 Vector（此块覆盖写，不混入旧 acc）
        PV_READY / PV_RELEASE：确保 Vector 消费完成前不覆写 pv
        Vector：acc = alpha * acc + pv；推进 m/l
    Vector：有效行 O=acc/l，转 FP16，按 batch 和 Q 行偏移 store
```

上述 READY/RELEASE 是依赖名称，实际事件号须按槽位预算并配对。源码中的两个 section 不能代替逐块双向握手；历史 acc 的缩放是 Vector 操作，不是直接在 L0C 上调用 Tile mul。

源码参照（PyPTO 仓内）：`python/tests/st/pypto_pro/frontend/fa/test_fa_perf_tkv_preload_dn_vf_tail_tilegroup.py`。只作源码走读，不作为课程运行依赖；它包含的高级布局和预取不能省略后仍称作完整 Kernel。


## 2. 三个关键片段（Kernel 内片段，需先分配 Tile）

**片段一：online 更新**。s_blk 是已乘 `1/sqrt(H)` 并应用 mask 的 FP32 logits；统计 Tile 为 [本 AIV 行数,1] DN，广播以此为输入。逐元素统计运算需按对应 API 的布局规则使用一致视图，并对别名同步。

```python
pl.maximum(m_block, s_blk, tmp, dim=0)   # 当前块行最大值
pl.maximum(m_new, m_old, m_block)       # 必须合并历史最大值
pl.sub(diff, m_old, m_new)              # 同形状行状态相减，不做广播
pl.exp(alpha, diff)
pl.mul(l_new, l_old, alpha)
pl.expand_sub(shifted, s_blk, m_new, dim=0)
pl.exp(p_blk, shifted)                  # FP32，供求和；PV 输入转换另做
pl.sum(p_sum, p_blk, tmp, dim=0)
pl.add(l_new, l_new, p_sum)
pl.expand_mul(acc, acc, alpha, dim=0)   # [rows,H] × [rows,1]，必须广播
# Cube 计算 p@V，跨核搬回 pv_ub 并等待就绪后：
pl.add(acc, acc, pv_ub)
# 随后推进 m_old/l_old 到 m_new/l_new，不能遗漏状态更新
```

此片段假设本行已有或本块含有效 logits。第一次有有效块时令 alpha=0；此前没有有效列且本块全遮蔽时，保持 l=0/acc=0，避免 `-inf-(-inf)`。NaN 一旦进入状态量，最后的除法无法修复。PV 若把 p 转 FP16，属于额外舍入点，需单独证明满足 RFC 阈值。

**片段二：causal mask**。令 q0/q1、k0/k1 为当前块全局起止位置（右开）：

```text
k1 <= q0+1：全可见。
k0 >= q1：整块遮蔽，可跳过；所有参与端须一致跳过事件/计算。
其他：逐元素判断 key_position <= query_position，否则 logit=-inf。
```

valid_shape 只处理矩形尾块，不能表示三角 mask。对角块中每行可见列数不同，m/l 更新前必须应用 mask。

**片段三：最终归一化**。

```python
# 已确认本行 l>0；l=0 的行按明确契约直接写 0
pl.expand_div(o_blk, acc, l_final, dim=0)
# cast FP16 → store；输出坐标须包含 batch 和 Q 行偏移
```

本章无 mask 或标准等长 causal、S>0 时，每行至少有一个有效 key；零有效列分支主要为第 11 章 causal+选择的组合预留。


## 3. 从串行基线到流水

下文完整参考先交付逐块串行的 C(QK)→V(online/p)→C(PV)→V(acc)。四个事件依次为 score 就绪、p 就绪、PV 就绪、本轮累加结束；最后一个事件被 Cube 消费后才能覆写 GM 槽位。两个 AIV 即使本轮没有有效输出行，也参与握手，避免参与端数量不一致。

后续优化：① 将 GM 中转缩短为官方片上数据路径；② 为 QK/P/PV 分别规划多槽和反向释放；③ 在保持 online 状态跨轮依赖的前提下引入预取或自动 stage；④ 最后评估 Phase。每次只改一项并回归。四个阶段共享 Cube/Vector 资源，不能直接套用各阶段彼此独立时的吞吐公式。


### 完整基线的三个阅读断点

1. `build_attention(H)`：H 作为编译期常量，B/S 动态；按 `(batch,Q块)` 分任务。K 转置搬入使用原 Tensor 的轴 `[2,1]`，L1 布局为 ZN。
2. `mask_block`：使用 q0/k0 的原始 token 位置。有限哨兵 -1e30 避免空行产生 `-inf-(-inf)`，exp 后再把无效位置严格置 0；在本例有限 FP16 输入与 H≤128 的范围内，该哨兵小于所有合法 logits。
3. `prepare_attention`：分配每组 workspace，检查选块唯一性，返回可反复启动的闭包。非 128 整除的 S 在 Host 将 Q/K/V 补零到整块，Kernel 从 out.shape[1] 取得原始 S 并屏蔽补齐列；PV 的归约轴始终为 128，不假设分形无效区自动为零。参数检查、索引读取、补零和分配在闭包外，性能计时应包围预热后的闭包调用，并另记端到端准备成本。

完整源文件包含 Kernel、Host 包装和验收。它保持 m/l/acc 为 FP32，PV 的 p 输入转 FP16 是明确舍入点，仍须达到 1e-3 阈值。


In [ ]:
!cat ./src/sdpa_reference.py


## 4. 课后练习

1. （填空）online 更新的三个运行量：______（行最大值）、______（指数和）、______（输出累加矩阵）；每块的修正系数是 ______。
2. （实践）对照官方 FlashAttention 示例（cube_vector_fusion 文档进阶篇）与本节走读，列出三处官方实现比教学版"多做的事"（提示：预取、VF 函数、mask 的处理细节）。

In [ ]:
!cat ./answer/10.04_answer.txt

## 5. 本节小结

- 结构 = **Q 块任务（核间）× K 块循环（核内）× Cube/Vector 块级交替**；
- online 三运行量（m/l/acc）+ 修正系数 exp(m_old-m_new) 贯穿全循环；
- causal 的块级三分法（全可见/全遮蔽/跨边界）；
- 教学版正确优先 → 手动流水 → 自动流水 → Phase，调优路径与第 8 章一致。